<a href="https://colab.research.google.com/github/pedraocoder/Projeto-AV2-ML1-ET0-Pernambuco/blob/main/et0_pernambuco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# PARTE 1: Carregamento dos dados e visualização inicial:

import pandas as pd
import numpy as np
import os
import zipfile

with zipfile.ZipFile('Dados_inmet_pernambuco_2023.zip', 'r') as z:
    z.extractall('dados_inmet')

def ler_estacao(filepath):
    meta = {}
    with open(filepath, encoding='latin-1') as f:
        for i, line in enumerate(f):
            if i == 0: meta['nome']      = line.split(':')[1].strip()
            if i == 1: meta['codigo']    = line.split(':')[1].strip()
            if i == 2: meta['latitude']  = float(line.split(':')[1].strip())
            if i == 3: meta['longitude'] = float(line.split(':')[1].strip())
            if i == 4: meta['altitude']  = float(line.split(':')[1].strip())
            if i == 9: break

    df = pd.read_csv(filepath, sep=';', skiprows=10,
                     encoding='latin-1', decimal='.',
                     na_values=['null', 'NULL', '', ' '])
    df.columns = ['data','pressao','tmax','tmed',
                  'tmin','ur_med','ur_min','vento','vazio']
    df = df.drop(columns=['vazio'], errors='ignore')
    df['data'] = pd.to_datetime(df['data'], errors='coerce')
    df = df.dropna(subset=['data'])
    for k, v in meta.items():
        df[k] = v
    return df

# Nota: 3 estações (Caruaru, Petrolina, Serra Talhada) tiveram falha
# de equipamento no início de 2023 e só possuem dados parciais.
# Estações com menos de 30 dias válidos são descartadas automaticamente.

todos = []
print("Carregando estações...\n")
for root, dirs, arquivos in os.walk('dados_inmet'):
    for arq in sorted(arquivos):
        if arq.endswith('.csv'):
            est = ler_estacao(os.path.join(root, arq))
            dias_validos = est['tmed'].notna().sum()
            if dias_validos >= 30:
                est = est.dropna(subset=['tmax','tmed','tmin','ur_med','vento'], how='all')
                todos.append(est)
                print(f" {est['nome'].iloc[0]:<15} {dias_validos} dias válidos")
            else:
                print(f" {est['nome'].iloc[0]:<15} sem dados — ignorada")

df = pd.concat(todos, ignore_index=True)
print(f"\n{'='*50}")
print(f"Total de estações carregadas: {df['nome'].nunique()}")
print(f"Total de registros:           {len(df)}")
print(f"Período:                      {df['data'].min().date()} a {df['data'].max().date()}")

print(f"\n--- Estações e coordenadas ---")
print(df[['nome','codigo','latitude','longitude','altitude']]
      .drop_duplicates()
      .sort_values('nome')
      .to_string(index=False))

print(f"\n--- Primeiros registros com dados válidos ---")
print(df[df['tmed'].notna()][['nome','data','tmax','tmed','tmin','ur_med','vento']]
      .head(10)
      .to_string(index=False))

print(f"\n--- Tipos das colunas ---")
print(df.dtypes)

Carregando estações...

  ✓ PETROLINA       239 dias válidos
  ✓ ARCO VERDE      334 dias válidos
  ✓ GARANHUNS       362 dias válidos
  ✓ SURUBIM         312 dias válidos
  ✓ CABROBO         365 dias válidos
  ✓ CARUARU         235 dias válidos
  ✓ IBIMIRIM        365 dias válidos
  ✓ SERRA TALHADA   305 dias válidos
  ✓ FLORESTA        365 dias válidos
  ✓ PALMARES        365 dias válidos
  ✓ OURICURI        197 dias válidos
  ✓ SALGUEIRO       333 dias válidos

Total de estações carregadas: 12
Total de registros:           3835
Período:                      2023-01-01 a 2023-12-31

--- Estações e coordenadas ---
         nome codigo  latitude  longitude  altitude
   ARCO VERDE   A309 -8.433611 -37.055556    683.95
      CABROBO   A329 -8.503889 -39.315278    342.74
      CARUARU   A341 -8.356111 -36.028333    837.00
     FLORESTA   A351 -8.598889 -38.584167    327.42
    GARANHUNS   A322 -8.910833 -36.493333    827.78
     IBIMIRIM   A349 -8.509444 -37.711667    434.23
     OURICURI